# Reproducing Manuscript 2 in Google Colab

This notebook reproduces every table, figure and number of Manuscript 2 that derives from the deposited final-test predictions:

*When Model Choice Changes with the Seed: Decision-State Uncertainty and Break-Even Costs in Autoencoder Intrusion Detection*

I wrote it for Google Colab because Experiment 1 v2 ran there. It needs no GPU, because nothing is trained: every step is arithmetic and statistics on the frozen predictions.

**What you need to do:** run the cells from top to bottom. When the last check cell prints `233 of 233 manuscript values reproduced`, the reproduction is complete.

## 1. Get the code

Set `REPO_URL` to the address of this repository, then run the cell.

In [ ]:
REPO_URL = "https://github.com/<account>/<repository>.git"   # replace with this repository's address
!git clone --depth 1 {REPO_URL} repo
%cd repo

## 2. (Optional) Use the exact library versions

Colab's preinstalled libraries are usually newer than the ones I used. With newer versions, the reproduced numbers can differ in the last floating-point digits: I measured relative differences of at most 2 × 10⁻¹³. Every manuscript value is still reproduced.

If you want byte-identical output files, uncomment and run this cell, then restart the runtime from **Runtime → Restart session** and continue from Section 1 again.

In [ ]:
# import sys
# !{sys.executable} -m pip install -q -r requirements.txt

## 3. Download the deposited predictions from Zenodo

The 20 Parquet files are deposited on Zenodo (doi:10.5281/zenodo.23559367, CC BY 4.0). The cell below uses the `zenodo_get` command-line tool to download the record into `data/raw/`.

If the download fails, download the Parquet files by hand from the Zenodo record page and upload them to `data/raw/`. Alternatively, mount Google Drive and point `DATA_DIR` at the folder that holds them.

File names do not matter: the analysis identifies each file by its MD5 checksum against `data/input_manifest.csv`.

In [ ]:
!pip install -q zenodo_get
!mkdir -p data/raw
!zenodo_get 10.5281/zenodo.23559367 -o data/raw
DATA_DIR = "data/raw"

In [ ]:
# Alternative: files kept on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/<folder with the 20 Parquet files>"

## 4. Run the full reproduction

The script authenticates the inputs, checks the paired design and the prediction rule, recomputes all results, draws the figures and compares every value with the manuscript. It writes everything to `results/`. On a standard Colab CPU runtime it takes under a minute.

In [ ]:
import sys
!{sys.executable} scripts/reproduce_all.py --data-dir {DATA_DIR} --out-dir results

## 5. Inspect the reproduced tables and figures

In [ ]:
import pandas as pd
from IPython.display import display, Image
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
for name in ["table_4_1_five_seed_performance", "table_4_2_seed_decision_states", "table_4_3_path_composition",
             "table_4_4_decision_map_unsw_nb15", "table_4_5_score_diagnostics", "mcnemar_by_seed"]:
    print(name); display(pd.read_csv(f"results/tables/{name}.csv"))

In [ ]:
for stem in ["fig4_1_decision_state_plane", "fig4_2_path_composition", "fig4_3_cost_ratio_staircase"]:
    display(Image(f"results/figures/{stem}.png", width=700))

## 6. Check against the manuscript

Each row of `results/manuscript_comparison.csv` is one value printed in the manuscript, with the place it is printed, the printed value and the reproduced value.

In [ ]:
cmp = pd.read_csv("results/manuscript_comparison.csv")
print(f"{int(cmp['match'].sum())} of {len(cmp)} manuscript values reproduced")
display(cmp[~cmp["match"]] if (~cmp["match"]).any() else cmp.head(10))